In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql import DataFrame
from functools import reduce
from delta.tables import DeltaTable
import requests, zipfile, io
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
from datetime import datetime
import pprint
import os
import json
import time

## CVM - Informações Diárias 

### 1. Verificando os arquivos

Essa logicaaa prioriza dos de M-0 e M-1, devido a atualização na base origem. Casos os dados forem de meses M-2 + serão atualizado aos domingos quando a base origem atualiza. Importante logica para controle de carga da pipeline e para FinOps.

In [0]:
BASE_URL = "https://dados.cvm.gov.br/dados/FI/DOC/INF_DIARIO/DADOS/"

response = requests.get(BASE_URL)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# regex para pegar apenas arquivos de 2026
pattern = re.compile(r"inf_diario_fi_(\d{6})\.zip")

hoje = datetime.today()
current_year_month = hoje.strftime("%Y%m")
current_month = hoje.month
current_year = hoje.year

dia_da_semana_update = hoje.weekday() == 6 # Domingo


files = []

for link in soup.find_all("a", href=True):
    href = link["href"]
    match = pattern.match(href)

    if match:
        file_yyyymm = match.group(1)
        file_year = int(file_yyyymm[:4])
        file_month = int(file_yyyymm[4:6])
        
        diff_month = (current_year - file_year) * 12 + (current_month - file_month)
        # ========================================== 
        #           REGRA DE ATUALIZAÇÃO
        #===========================================
        # M-0 e M-1 > atualizam diariamente
        # M-2 + > Atualizam semanalmente nos domingos 


        
        if diff_month <= 1:
            files.append(urljoin(BASE_URL, href))

        elif diff_month >= 2 and dia_da_semana_update:
            files.append(urljoin(BASE_URL, href))


print(files)

### 2. Extraindo os arquivos

In [0]:

for zip_url in files:
    print(f'Processando: {zip_url}')

    response = requests.get(zip_url)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        for file_name in z.namelist():
            if file_name.endswith(".csv"):
                output_path = os.path.join("/Volumes/workspace/case_spark_cvm/raw/cvm_informe_diario/", file_name)

                with z.open(file_name) as source, open(output_path, "wb") as target:
                    target.write(source.read())

                print(f"  → Extraído: {output_path}")


### 3. Salvar em camada Bronze Particionada

Nesta etapa, realizamos a ***normalização*** das colunas. Nos arquivos recentes, foi incluída a coluna **ID_SUBCLASSE**; já nos registros de anos anteriores, essa coluna é inexistente e a nomenclatura das demais difere do padrão atual. Esse processo garante a padronização e a ordenação ***correta*** dos dados para o consumo.

In [0]:
caminho_raw = "/Volumes/workspace/case_spark_cvm/raw/cvm_informe_diario/"

# 1. Lista todos os arquivos CSV dentro do volume
arquivos_raw = [file.path for file in dbutils.fs.ls(caminho_raw) if file.name.endswith('.csv')]

lista_dfs = []

# Ordem oficial que queremos na nossa tabela Bronze
colunas_ordem = [
    "TP_FUNDO_CLASSE", "CNPJ_FUNDO_CLASSE", "ID_SUBCLASSE",
    "DT_COMPTC", "VL_TOTAL", "VL_QUOTA", "VL_PATRIM_LIQ",
    "CAPTC_DIA", "RESG_DIA", "NR_COTST"
]

# 2. Lê e padroniza cada arquivo
for arquivo in arquivos_raw:
    df_temp = spark.read.csv(arquivo, sep=';', header=True)
    
    # Se detectar que é o layout antigo (tem TP_FUNDO em vez de TP_FUNDO_CLASSE)
    if "TP_FUNDO" in df_temp.columns:
        df_temp = df_temp \
            .withColumnRenamed("TP_FUNDO", "TP_FUNDO_CLASSE") \
            .withColumnRenamed("CNPJ_FUNDO", "CNPJ_FUNDO_CLASSE") \
            .withColumn("ID_SUBCLASSE", f.lit(None).cast("string")) # Cria a coluna faltante como nula
            
    # Garante que todos os dataframes tenham as colunas na mesma ordem
    df_temp = df_temp.select(*colunas_ordem)
    
    lista_dfs.append(df_temp)

# 3. Une todos os DataFrames corretamente (agora todos têm as mesmas 10 colunas)
df_cvm = reduce(DataFrame.unionByName, lista_dfs)

# 4. Adiciona a data de processamento
df_cvm = df_cvm.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

# 5. Salva na camada Bronze
data_proc = int(datetime.now().strftime("%Y%m%d"))

df_cvm.write \
    .mode("overwrite") \
    .option("replaceWhere", f"data_processamento = {data_proc}") \
    .partitionBy("data_processamento") \
    .format("delta") \
    .save("/Volumes/workspace/case_spark_cvm/bronze/cvm_informe_diario/")

In [0]:
display(df_cvm)